In [ ]:
from fastai.torch_core import to_detach as _orig_to_detach
import fastai.learner as _fl
import fastai.torch_core as _ftc
from fastai.learner import Recorder as _Recorder
from fastai.callback.tracker import SaveModelCallback as _SaveModelCallback
from fastai.learner import AvgSmoothLoss as _AvgSmoothLoss
from fastai.callback.progress import ProgressCallback as _ProgressCallback

In [ ]:
_orig_load_model = _fl.load_model

def _dml_safe_load_model(file, model, opt, with_opt=True, device=None, strict=True, **kwargs):
    """
    DirectML-safe load_model replacement.

    Problems solved:
    1. DirectML can't handle map_location=<torch.device> - load to CPU first
    2. Optimizer state (grad_avg, sqr_avg) lands on CPU after load - move to DML
    3. Lookahead slow_weights must match their corresponding fast param device
       exactly - all on DML
    """
    # Step 1: load everything to CPU
    _orig_load_model(file, model, opt, with_opt=with_opt,
                     device=torch.device('cpu'), strict=strict, **kwargs)

    # Step 2: resolve target device
    if device is not None:
        target_device = device
    else:
        param = next(
            (p for name, p in model.named_parameters() if 'out_proj' not in name),
            None
        )
        target_device = param.device if param is not None else torch.device('cpu')

    # Step 3: move model to DML
    model.to(target_device)

    # Step 4: move inner Adam state to DML
    # opt is Lookahead wrapper - inner Adam is opt.opt
    if opt is not None:
        inner_opt = opt.opt if hasattr(opt, 'opt') else opt
        if hasattr(inner_opt, 'state'):
            for p, state in inner_opt.state.items():
                keys_to_update = []
                for k, v in state.items():
                    if isinstance(v, torch.Tensor):
                        state[k] = v.to(p.device)   # match param device, not target_device
                        keys_to_update.append(k)

                # Delete old CPU tensors explicitly
                for k in keys_to_update:
                    if hasattr(state[k], 'device') and str(state[k].device) == 'cpu':
                        del state[k]


        # Step 5: move Lookahead slow_weights - match each to its fast param device
        # param_lists and slow_weights have identical structure: L(L(tensor))
        # out_proj fast param is on CPU -> its slow weight must stay on CPU
        # all other fast params are on DML -> their slow weights go to DML
        if hasattr(opt, 'slow_weights') and opt.slow_weights is not None:
            try:
                moved = L()
                for slow_pg, fast_pg in zip(opt.slow_weights, opt.param_lists):
                    moved_pg = L()
                    for slow_w, fast_p in zip(slow_pg, fast_pg):
                        moved_pg.append(slow_w.to(fast_p.device))
                    moved.append(moved_pg)
                opt.slow_weights = moved

                # Verify no mismatches remain
                mismatches = [
                    (slow_w.device, fast_p.device)
                    for slow_pg, fast_pg in zip(opt.slow_weights, opt.param_lists)
                    for slow_w, fast_p in zip(slow_pg, fast_pg)
                    if str(slow_w.device) != str(fast_p.device)
                ]
                if mismatches:
                    raise RuntimeError(f"slow_weights device mismatch: {mismatches}")

                print(f"\n[_dml_safe_load_model] slow_weights moved OK - "
                      f"{sum(len(pg) for pg in opt.slow_weights)} tensors, "
                      f"count={opt.count}, next_sync={opt.k - (opt.count % opt.k)}")

            except Exception as e:
                print(f"[_dml_safe_load_model] slow_weights move failed ({e}), resetting to None")
                opt.slow_weights = None
                opt.count = 0

_fl.load_model = _dml_safe_load_model

In [ ]:
_orig_recorder_after_epoch = _Recorder.after_epoch

def _patched_recorder_after_epoch(self):
    if len(self.log) < 3:  # epoch + train_loss + valid_loss minimum
        return
    _orig_recorder_after_epoch(self)

_Recorder.after_epoch = _patched_recorder_after_epoch

In [ ]:
def _patched_avg_smooth_loss_reset(self):
    self.count = 0
    self._val_float = 0.0
    self._cached_value = 0.0  # cache avoids double .item() per batch

def _patched_avg_smooth_loss_accumulate(self, learn):
    self.count += 1
    # learn.loss is already a scalar - .mean() dispatches a pointless DML kernel
    # use .detach().item() directly
    loss_scalar = learn.loss.detach().item()
    self._val_float = self._val_float * self.beta + loss_scalar * (1.0 - self.beta)
    bias = 1.0 / (1.0 - self.beta ** self.count)
    self._cached_value = self._val_float * bias

@property
def _patched_avg_smooth_loss_value(self):
    # Returns Python float - downstream .item() calls become no-ops
    # recorder.losses stores floats not tensors -> RAM not VRAM
    return self._cached_value

_AvgSmoothLoss.reset      = _patched_avg_smooth_loss_reset
_AvgSmoothLoss.accumulate = _patched_avg_smooth_loss_accumulate
_AvgSmoothLoss.value      = _patched_avg_smooth_loss_value

In [ ]:
_orig_progress_after_batch = _ProgressCallback.after_batch

def _patched_progress_after_batch(self):
    self.pbar.update(self.iter + 1)
    if hasattr(self, 'smooth_loss'):
        sl = self.smooth_loss
        # smooth_loss.value now returns Python float - .item() not available
        val = sl if isinstance(sl, float) else sl.item()
        self.pbar.comment = f'{val:.4f}'

_ProgressCallback.after_batch = _patched_progress_after_batch

In [ ]:
def _dml_to_detach(b, cpu=True, gather=True):
    """
    Skip cpu=True on DirectML - moving large tensors (B,1,T) from DML
    to CPU costs 121ms per call (confirmed via cProfile).
    Detach still happens so gradients don't accumulate.
    """
    def _inner(x):
        if not isinstance(x, torch.Tensor):
            return x
        x = x.detach()
        if cpu and x.device.type not in ('privateuseone',):
            x = x.cpu()
        return x
    from fastai.torch_core import apply
    return apply(_inner, b)

# patch everywhere fastai uses to_detach
_ftc.to_detach = _dml_to_detach
_fl.to_detach  = _dml_to_detach

In [ ]:
class SafeSaveModelCallback(_SaveModelCallback):
    """
    Guards against two resume failures in SaveModelCallback:
    
    1. IndexError in after_epoch: recorder.values[-1] exists but is shorter
       than self.idx - happens when start_epoch > 0 and first epoch hasn't
       fully populated recorder yet
       
    2. IndexError in after_fit: learn.load() inside after_fit hits same issue
       when loading best model at end of training
    """
    def after_epoch(self):
        vals = self.learn.recorder.values
        if not vals:
            return
        if len(vals[-1]) <= self.idx:
            return
        super().after_epoch()

    def after_fit(self, **kwargs):
        try:
            super().after_fit(**kwargs)
        except (IndexError, Exception) as e:
            print(f"[SafeSaveModelCallback] after_fit skipped: {e}")